**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `python_prebuilt/` and the bundle's R routes (`R/lib`, see `R/README_BRIDGE.md`; the full R pipeline is the project RWDR). Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# V03 — Classical Estimator Simulation Validation
Runs every **classical** DiD estimator against simulated data with a **known true effect**.
Covers the 9 methods added in v5.0/v6.0 (stacked DiD, ETWFE, entropy balancing, Honest DiD,
quantile DiD, interactive FE, matrix completion, generalized synthetic control, ML-CATE).

Run this after editing any formula in `_common.py`.

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

sys.path.insert(0,"../All_DIDsCodes"); sys.path.insert(0,"All_DIDsCodes")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("V03")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob
C.step(1,"running validate_all() -- simulated ground truth for every classical estimator")
res=C.validate_all(verbose=True)
n=sum(1 for v in res.values() if v=="PASS")
(C.ok if n==len(res) else C.fail)(f"{n}/{len(res)} classical estimators validated")
pd.DataFrame([{"method":k,"result":v} for k,v in res.items()]).to_csv("V03_results.csv",index=False)
C.ok("saved -> V03_results.csv")
assert n==len(res), "one or more estimators failed validation -- do NOT use the pipeline until fixed"

# ---- MEMORY RELEASE (v14): free this model's frames so the next one starts clean ----
try:
    C.release(*[v for k,v in list(globals().items()) if k in ("panel","df","result") and v is not None])
except Exception as _e:
    print(f"[INFO]    release skipped: {_e}")
C.done(1, next_task="run V04_ML_Causal_Method_Validation.ipynb")
